In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F


spark = (
    SparkSession
    .builder
    .appName("fut3r_pyspark_local")
    .getOrCreate()
)

print("Активные Spark сессии:", spark.sparkContext.uiWebUrl)

Активные Spark сессии: http://e312d7bcc595:4040


In [2]:
from pyspark.sql import SparkSession
print(SparkSession.getActiveSession())

None


In [22]:
spark.stop()
print(SparkSession.getActiveSession())
#print("Активные Spark сессии:", spark.sparkContext.uiWebUrl)

None


In [1]:
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SparkExample") \
    .config("spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
        "ru.yandex.clickhouse:clickhouse-jdbc:0.3.2,"
        "org.postgresql:postgresql:42.5.0,"
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0",
        ) \
    .getOrCreate()


hadoop_conf = spark._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", os.getenv("MINIO_ROOT_USER"))
hadoop_conf.set("fs.s3a.secret.key", os.getenv("MINIO_ROOT_PASSWORD"))
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hadoop_conf.set("fs.s3a.path.style.access", "true")

:: loading settings :: url = jar:file:/usr/local/lib/python3.11/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
ru.yandex.clickhouse#clickhouse-jdbc added as a dependency
org.postgresql#postgresql added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-45c6e908-94d2-4a5e-958a-2724d9d225b6;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found ru.yandex.clickhouse#clickhouse-jdbc;0.3.2 in central
	found com.clickhouse#clickhouse-http-client;0.3.2 in central
	found com.clickhouse#clickhouse-client;0.3.2 in central
	found org.lz4#lz4-java;1.8.0 in central
	found com.google.code.gson#gson;2.8.8 in central
	found org.apache.httpcomponents#httpclient;4.5

In [2]:
# ⬇️ Параметры подключения к PostgreSQL public.shops 

jdbc_url = "jdbc:postgresql://postgres_source:5432/source" 
table_name = "public.shops" 
db_user = os.getenv("POSTGRES_USER")
db_password = os.getenv("POSTGRES_PASSWORD")


shops_df = spark.read \
		.format("jdbc")\
		.option("url", jdbc_url)\
		.option("user", db_user)\
		.option("password", db_password)\
		.option("dbtable", table_name)\
		.option("fetchsize", 1000)\
		.option("driver", "org.postgresql.Driver")\
		.load() 


shops_df.show()

+-----+-----------+
|st_id|  shop_name|
+-----+-----------+
|  842|      Lenta|
|  843|     Magnit|
|  844|       Spar|
|  845|Pyaterochka|
|  846|      Lenta|
|  847|      Diksi|
|  848|      Lenta|
|  849|   FixPrice|
|  850|     Magnit|
|  851|      Lenta|
+-----+-----------+



In [9]:
shops_df.show(20, truncate=False)

+-----+-----------+
|st_id|shop_name  |
+-----+-----------+
|842  |Lenta      |
|843  |Magnit     |
|844  |Spar       |
|845  |Pyaterochka|
|846  |Lenta      |
|847  |Diksi      |
|848  |Lenta      |
|849  |FixPrice   |
|850  |Magnit     |
|851  |Lenta      |
+-----+-----------+



In [11]:
shops_df.dtypes

[('st_id', 'int'), ('shop_name', 'string')]

In [3]:
# ⬇️ Параметры подключения к PostgreSQLpublic.shop_timezone 


jdbc_url = "jdbc:postgresql://postgres_source:5432/source" 
table_name = "public.shop_timezone" 
db_user = os.getenv("POSTGRES_USER") 
db_password = os.getenv("POSTGRES_PASSWORD") 


shop_timezone_df = spark.read \
		.format("jdbc")\
		.option("url", jdbc_url)\
		.option("user", db_user)\
		.option("password", db_password)\
		.option("dbtable", table_name)\
		.option("fetchsize", 1000)\
		.option("driver", "org.postgresql.Driver")\
		.load() 
                
shop_timezone_df.show()

+-----+---------+
|plant|time_zone|
+-----+---------+
|  842|         |
|  842|    RUS07|
|  843|    RUS04|
|  844|         |
|  845|         |
|  845|    RUS05|
|  847|    RUS03|
|  848|    RUS08|
|  848|         |
| P847|         |
| E103|    RUS08|
| -134|    RUS04|
|    0|         |
|    0|    RUS08|
|  848|         |
+-----+---------+



In [8]:
shop_timezone_df.where('plant=844').show()

+-----+---------+
|plant|time_zone|
+-----+---------+
|  844|         |
+-----+---------+



In [14]:
shop_timezone_df.show()

+-----+---------+
|plant|time_zone|
+-----+---------+
|  842|         |
|  842|    RUS07|
|  843|    RUS04|
|  844|         |
|  845|         |
|  845|    RUS05|
|  847|    RUS03|
|  848|    RUS08|
|  848|         |
| P847|         |
| E103|    RUS08|
| -134|    RUS04|
|    0|         |
|    0|    RUS08|
|  848|         |
+-----+---------+



In [4]:
import pyspark.sql.functions as F
'''shop_timezone_df = (
    shop_timezone_df.na
    .replace('', '3', 'time_zone')
)'''

shops_df.show()

tz_df = (
    shop_timezone_df.select(
        F.col("plant").cast("string"),
        F.coalesce(
            F.when(F.col("time_zone").rlike(r"RUS0\d+"), F.regexp_extract(F.col("time_zone"), r"RUS0(\d+)", 1)),
            F.lit(3)
         ).cast("int").alias('time_zone')
    )
)

tz_df.show()

+-----+-----------+
|st_id|  shop_name|
+-----+-----------+
|  842|      Lenta|
|  843|     Magnit|
|  844|       Spar|
|  845|Pyaterochka|
|  846|      Lenta|
|  847|      Diksi|
|  848|      Lenta|
|  849|   FixPrice|
|  850|     Magnit|
|  851|      Lenta|
+-----+-----------+

+-----+---------+
|plant|time_zone|
+-----+---------+
|  842|        3|
|  842|        7|
|  843|        4|
|  844|        3|
|  845|        3|
|  845|        5|
|  847|        3|
|  848|        8|
|  848|        3|
| P847|        3|
| E103|        8|
| -134|        4|
|    0|        3|
|    0|        8|
|  848|        3|
+-----+---------+



In [5]:
res_df = (
    shops_df.join(tz_df, shops_df.st_id == tz_df.plant)
   .groupBy('st_id','shop_name')
   .agg(F.max('time_zone').alias('tz_code'))
   .show()
)

+-----+-----------+-------+
|st_id|  shop_name|tz_code|
+-----+-----------+-------+
|  842|      Lenta|      7|
|  843|     Magnit|      4|
|  844|       Spar|      3|
|  845|Pyaterochka|      5|
|  847|      Diksi|      3|
|  848|      Lenta|      8|
+-----+-----------+-------+

